# 02 - Model adapters: equivalence with FinBench

Step 2 of `THESIS_GUIDE.md`. The six models are wrapped behind one interface, `xaifin.models.base.ModelAdapter`, so that training, explanations, portfolios and the app can use any of them the same way. This notebook checks that **each adapter trains and predicts exactly as FinBench's own code does**, so that everything computed later through the adapters stands for the FinBench models.

**What is compared, for each model** (DJI, test year 2020, T = 20, L = 5, seed 42, `CLEAN_DATA = False` because FinBench keeps the bad prices):

1. **Initial weights**: FinBench's model and the adapter's, both built after seeding every generator with the same seed.
2. **Training**: from the same initial weights, one epoch over the training days, in the same order and with the same random state before each day (dropout, sampling noise). Then the largest difference between the two sets of weights.
3. **Test predictions and metrics**: each side predicts the test year with its own evaluation code; largest difference in predictions and labels, and the metrics of both.

**How FinBench's code is run.** The training scripts cannot run here unchanged: they read `../../Evaluation/data` and import libraries that are not installed (qlib, wandb, tensorboard, torch_geometric, audtorch). `finbench_reference.py`, next to this notebook, loads FinBench's files read-only and calls FinBench's own **data code, model classes, training-step and evaluation functions** (for example `SequenceModel.train_epoch` and `predict` for MASTER, `train_epoch` / `valid_epoch` of MATCC's `train.py`). The only replacement is FinFormer's `edgeIndexTransform`, which needs torch_geometric (same edges, checked when the adapter was written).

**Same randomness on both sides.** FinBench draws dropout and sampling noise from one random stream, and does not seed it at all for MASTER, MATCC and FinFormer (issue 8 in `docs/model_notes.md`). Here both sides go through the same driver (`SeededDays`), which fixes the order of the days and reseeds torch before each one.

**Expected exceptions** (decided fixes, `docs/model_notes.md`): FactorVAE's test predictions. FinBench scores the reconstruction from the *posterior* factors, which reads the true labels (issue 1), and samples it (issue 2); the adapter predicts the mean of the *prior* path. Its predictions are compared with FinBench's `prediction()` without the sampling noise instead.

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # deterministic cuBLAS, set before CUDA starts

import contextlib, copy, io, time, warnings
import numpy as np
import pandas as pd
import torch

from finbench_reference import (DiscoverPLFReference, FactorVAEReference, FinFormerReference, HISTReference,
                                MASTERReference, MATCCReference, SeededDays, state_diff)
from xaifin.config import rolling_window
from xaifin.models.base import RunConfig, set_seed
from xaifin.models.discoverplf import DiscoverPLFAdapter
from xaifin.models.factorvae import FactorVAEAdapter
from xaifin.models.finformer import FinFormerAdapter
from xaifin.models.hist import HISTAdapter
from xaifin.models.master import MASTERAdapter
from xaifin.models.matcc import MATCCAdapter
from xaifin.training.metrics import summary
from xaifin.training.trainer import evaluate, train_epoch

warnings.filterwarnings("ignore", category=FutureWarning)
torch.use_deterministic_algorithms(True, warn_only=True)

UNIVERSE, TEST_YEAR, SEED = "dji", 2020, 42
WINDOW = rolling_window(TEST_YEAR)
BASE_SEED = 10_000  # torch is reseeded with BASE_SEED + k before the k-th training day, on both sides
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "| torch", torch.__version__)

NVIDIA GeForce RTX 5090 | torch 2.11.0+cu128


In [2]:
def largest_difference(a: list, b: list) -> float:
    return max(float(np.abs(np.ravel(x) - np.ravel(y)).max()) for x, y in zip(a, b))


def compare(Adapter, Reference) -> dict:
    # Builds both sides, trains one epoch on each, and returns the differences and both test metrics.
    start = time.time()
    adapter = Adapter(RunConfig(Adapter.name, UNIVERSE, TEST_YEAR, seed=SEED, clean=False))
    with contextlib.redirect_stdout(io.StringIO()):  # FinBench's data code prints its progress
        ref = Reference(UNIVERSE, WINDOW, adapter.cfg.seq_len, adapter.cfg.pred_len)

    set_seed(SEED)
    init_diff = state_diff(ref.init_state(), adapter.model.state_dict())

    train = adapter.splits()["train"]
    order = np.random.default_rng(0).permutation(len(train))
    ref.load(copy.deepcopy(adapter.model.state_dict()))
    with contextlib.redirect_stdout(io.StringIO()):
        ref.train_epoch(order, BASE_SEED)
    optimizer, scheduler = adapter.configure_optimizer()
    train_epoch(adapter, optimizer, scheduler, epoch=0, days=SeededDays(train.__getitem__, order, BASE_SEED))
    weight_diff = state_diff(ref.state(), adapter.model.state_dict())

    with contextlib.redirect_stdout(io.StringIO()):
        fb_preds, fb_labels = ref.predict_test()
    preds, labels, metrics = evaluate(adapter, "test")
    fb_metrics = summary(fb_preds, fb_labels)
    row = {
        "model": Adapter.name,
        "training days": len(order),
        "test days": len(preds),
        "initial weights": init_diff,
        "weights after 1 epoch": weight_diff,
        "test predictions": largest_difference(fb_preds, preds),
        "test labels": largest_difference(fb_labels, labels),
        "FinBench RankIC": fb_metrics["RankIC"],
        "adapter RankIC": metrics["RankIC"],
        "FinBench IC": fb_metrics["IC"],
        "adapter IC": metrics["IC"],
        "FinBench MSE": fb_metrics["MSE"],
        "adapter MSE": metrics["MSE"],
        "seconds": round(time.time() - start),
    }
    return row, adapter, ref, preds


def show(row: dict) -> pd.DataFrame:
    return pd.DataFrame([row]).set_index("model").T


rows = {}

## 1. MASTER

FinBench: `Regression/MASTER/` — `load_dataset.CSVDataset`, `master.MASTERModel` with `SequenceModel.train_epoch` (drop the 2.5% label tails, daily z-score, MSE, Adam, gradient clipping at 3) and `SequenceModel.predict`. Market gate features: 63 for the US.

In [3]:
row, adapter, ref, preds = compare(MASTERAdapter, MASTERReference)
rows["MASTER"] = row
show(row)

model,MASTER
training days,9.820000e+02
test days,2.530000e+02
initial weights,0.000000e+00
weights after 1 epoch,2.384186e-06
test predictions,7.152557e-07
test labels,4.768372e-07
FinBench RankIC,-7.592524e-03
adapter RankIC,-7.592524e-03
FinBench IC,-1.506684e-02
adapter IC,-1.506686e-02


## 2. MATCC

FinBench: `Regression/MATCC/` — `load_dataset.CSVDataset` (same as MASTER's), `model.MATCC.MATCC`, `lr_scheduler.ChainedScheduler`, and `train_epoch` / `valid_epoch` of `train.py` (Adam with weight decay, scheduler stepped once per epoch).

In [4]:
row, adapter, ref, preds = compare(MATCCAdapter, MATCCReference)
rows["MATCC"] = row
show(row)

model,MATCC
training days,9.820000e+02
test days,2.530000e+02
initial weights,0.000000e+00
weights after 1 epoch,4.484745e-05
test predictions,1.758337e-06
test labels,7.152557e-07
FinBench RankIC,9.852000e-03
adapter RankIC,9.852000e-03
FinBench IC,1.182151e-02
adapter IC,1.182153e-02


## 3. FactorVAE

FinBench: `Regression/FactorVAE/` — `dataset.init_data_loader` (Qlib's `TSDataSampler`, windows over the calendar with filled gaps), `module.FactorVAE`, `model.train` (VAE loss, cosine learning rate per day) and `model.test`. The label is the 20-day return (issue 3, kept). Test predictions are expected to differ: `model.test` scores the leaky posterior reconstruction (issue 1).

In [5]:
row, adapter, ref, preds = compare(FactorVAEAdapter, FactorVAEReference)
rows["FactorVAE"] = row
show(row)

model,FactorVAE
training days,1006.000000
test days,253.000000
initial weights,0.000000
weights after 1 epoch,0.000000
test predictions,0.848527
test labels,0.000000
FinBench RankIC,-0.007841
adapter RankIC,0.029651
FinBench IC,-0.009601
adapter IC,0.019137


In [6]:
# The adapter predicts the mean of the prior path (issues 1-2). FinBench's prediction() computes the same thing
# with a random sample; with its sampling switched off it must equal the adapter's predictions.
fb_mean = ref.predict_test_mean()
fb_labels = ref.predict_test()[1]
leak_free = summary(fb_mean, fb_labels)
rows["FactorVAE"]["test predictions"] = largest_difference(fb_mean, preds)
pd.DataFrame({
    "FinBench test() (posterior, reads the labels)": summary(*ref.predict_test()),
    "FinBench prediction() without sampling": leak_free,
    "adapter": summary(preds, fb_labels),
}).loc[["IC", "RankIC", "ICIR", "MSE"]]

,"FinBench test() (posterior, reads the labels)",FinBench prediction() without sampling,adapter
IC,-0.013161,0.019137,0.019137
RankIC,-0.009327,0.029651,0.029651
ICIR,-0.067400,0.077156,0.077156
MSE,0.971929,0.926780,0.926780


## 4. HIST

FinBench: `Regression/HIST/` — `create_loaders`, `train_epoch` and `test_epoch` of `train.py` (daily z-score of the label, MSE, gradient clipping at 3), `model.HIST`. Inputs: one Alpha360 row per stock, the stock-to-concept matrix and the market capitalization. FinBench's HIST only runs on a GPU (issue 9).

In [7]:
row, adapter, ref, preds = compare(HISTAdapter, HISTReference)
rows["HIST"] = row
show(row)

model,HIST
training days,1.001000e+03
test days,2.530000e+02
initial weights,0.000000e+00
weights after 1 epoch,0.000000e+00
test predictions,0.000000e+00
test labels,7.152557e-07
FinBench RankIC,2.952117e-02
adapter RankIC,2.952117e-02
FinBench IC,2.176848e-02
adapter IC,2.176848e-02


## 5. DiscoverPLF

FinBench: `Regression/DiscoverPLF/` — `create_loaders`, `train_epoch` and `test_epoch` of `train.py` (MSE plus 0.05 x the VAE losses, with the 'increasing' weight of epoch 0), `factormodel.FACTORMODEL`. Same data as HIST, except the constituents file.

In [8]:
row, adapter, ref, preds = compare(DiscoverPLFAdapter, DiscoverPLFReference)
rows["DiscoverPLF"] = row
show(row)

model,DiscoverPLF
training days,1.001000e+03
test days,2.530000e+02
initial weights,0.000000e+00
weights after 1 epoch,0.000000e+00
test predictions,0.000000e+00
test labels,7.152557e-07
FinBench RankIC,4.248114e-02
adapter RankIC,4.248114e-02
FinBench IC,2.865631e-02
adapter IC,2.865631e-02


## 6. FinFormer

FinBench: `Regression/FinFormer/` — `load_dataset.CustomDataset` (columns sorted alphabetically, reshaped to 60 x 6), `model.Finformer`, `Finformer_Model.trainer` / `tester` of `finformer.py` (loss = -CCC). The sector graph is rebuilt every day for the stocks of that day. FinBench moves the model to the GPU before its xavier initialization, so on a GPU the initial weights come from the CUDA generator; the adapter does the same.

In [9]:
row, adapter, ref, preds = compare(FinFormerAdapter, FinFormerReference)
rows["FinFormer"] = row
show(row)

D:\CodeProjects\master-thesis\code\finbench\Regression\FinFormer\model.py:84: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:656.)
  graph = torch.sparse.FloatTensor(indices=edge, values=value, size=attn[0].shape).to_dense().to(DEVICE)


model,FinFormer
training days,1001.000000
test days,253.000000
initial weights,0.000000
weights after 1 epoch,0.000000
test predictions,0.000000
test labels,0.000000
FinBench RankIC,0.018689
adapter RankIC,0.018689
FinBench IC,0.016209
adapter IC,0.016209


## 7. Summary

Pass criteria:
- **initial weights** identical (difference 0);
- **test predictions** within 1e-4 (float32 rounding), FactorVAE against FinBench's `prediction()` without sampling;
- **test metrics**: RankIC within 1e-4 of FinBench's (FactorVAE excepted, see section 3);
- **weights after one epoch** within 1e-3. A difference above float32 rounding is allowed here for one reason: on Adam's first steps every parameter moves by about ±lr (3e-4 for MATCC) whatever the size of its gradient, so a parameter whose gradient is pure rounding noise (about 1e-6, e.g. MATCC's DLinear biases) can step in either direction on the two sides. The predictions show whether this matters.

In [10]:
table = pd.DataFrame(rows.values()).set_index("model")
checks = pd.DataFrame({
    "initial weights": table["initial weights"] == 0,
    "weights after 1 epoch": table["weights after 1 epoch"] <= 1e-3,
    "test predictions": table["test predictions"] <= 1e-4,
    "test RankIC": ((table["FinBench RankIC"] - table["adapter RankIC"]).abs() <= 1e-4) | (table.index == "FactorVAE"),
})
table["passed"] = checks.all(axis=1)
pd.set_option("display.float_format", lambda v: f"{v:.2e}" if abs(v) < 1e-3 and v != 0 else f"{v:.4f}")
table

,training days,test days,initial weights,weights after 1 epoch,test predictions,test labels,FinBench RankIC,adapter RankIC,FinBench IC,adapter IC,FinBench MSE,adapter MSE,seconds,passed
model,,,,,,,,,,,,,,
MASTER,982,253,0.0000,2.38e-06,7.15e-07,4.77e-07,-0.0076,-0.0076,-0.0151,-0.0151,0.9994,0.9994,40,True
MATCC,982,253,0.0000,4.48e-05,1.76e-06,7.15e-07,0.0099,0.0099,0.0118,0.0118,1.0120,1.0120,44,True
FactorVAE,1006,253,0.0000,0.0000,0.0000,0.0000,-0.0078,0.0297,-0.0096,0.0191,0.9689,0.9268,315,True
HIST,1001,253,0.0000,0.0000,0.0000,7.15e-07,0.0295,0.0295,0.0218,0.0218,0.9660,0.9660,35,True
DiscoverPLF,1001,253,0.0000,0.0000,0.0000,7.15e-07,0.0425,0.0425,0.0287,0.0287,0.9816,0.9816,60,True
FinFormer,1001,253,0.0000,0.0000,0.0000,0.0000,0.0187,0.0187,0.0162,0.0162,2.3001,2.3001,41,True


In [11]:
# Done criterion of Step 2: adapter.forward(x, extras) returns [N] on a DayBatch, for every model.
for Adapter in (MASTERAdapter, MATCCAdapter, FactorVAEAdapter, HISTAdapter, DiscoverPLFAdapter, FinFormerAdapter):
    adapter = Adapter(RunConfig(Adapter.name, UNIVERSE, TEST_YEAR, seed=SEED))
    adapter.model.eval()
    batch = next(adapter.day_batches("test"))
    out = adapter.forward(batch.x, batch.extras)
    assert out.shape == (len(batch.tickers),)
    print(f"{Adapter.name:12s} x {tuple(batch.x.shape)} -> {tuple(out.shape)}, extras: {sorted(batch.extras) or '-'}")

MASTER       x (28, 20, 220) -> (28,), extras: -


MATCC        x (28, 20, 220) -> (28,), extras: -


FactorVAE    x (28, 20, 157) -> (28,), extras: -


HIST         x (28, 1, 360) -> (28,), extras: ['market_value', 'stock2concept']


DiscoverPLF  x (28, 1, 360) -> (28,), extras: ['market_value', 'stock2concept']


FinFormer    x (28, 1, 360) -> (28,), extras: ['adjacency']


## Observations

1. **All six adapters reproduce FinBench.** Under the same seed both sides build identical initial weights; after one full epoch (982-1,006 training days) with the same days and random state, the weights of FactorVAE, HIST, DiscoverPLF and FinFormer are bit-identical and those of MASTER and MATCC differ by 2e-6 and 4e-5; the test predictions differ by at most 2e-6 (FactorVAE against FinBench's leak-free `prediction()`), and IC, RankIC and MSE agree to the reported digits.
2. **MATCC's 4e-5 is float32 rounding, not a code difference.** It appears on the first training day and barely grows. The parameters involved (the DLinear biases) have a true gradient near 1e-6; the two sides compute it with different rounding, probably because FinBench slices the features out of a tensor that still holds the label column, and Adam's first updates move such parameters by about ±lr whatever the gradient's size. The adapter alone is deterministic (two runs: difference 0), and the predictions agree to 2e-6.
3. **FactorVAE: the readout fix is the only difference.** Training is bit-identical; FinBench's `prediction()` with its sampling switched off gives exactly the adapter's predictions. FinBench's own `test()` scores the posterior reconstruction, which reads the true labels (issue 1) and is a random sample (issue 2): its two calls in section 3 give different RankICs (-0.0078 and -0.0093) for the same weights. After one epoch the leaky score is not higher than the leak-free one (RankIC -0.009 vs 0.030); whether the leak inflates FinBench's FactorVAE results after full training is a Step 3 question.
4. **Seeding and device matter for reproducing FinBench's initial weights.** FinBench's MASTER, MATCC and FinFormer are not seeded (issue 8), so their published runs cannot be repeated; the adapters seed every generator. FinFormer is initialized on the device it runs on, as in FinBench, so its initial weights on a GPU differ from those on a CPU for the same seed.
5. **FinFormer's MSE is not comparable** (2.30 after one epoch, against about 1.0 for the others): its loss, -CCC, rewards correlation and agreement of mean and variance but not the scale FinBench measures with MSE. Compare FinFormer on IC and RankIC.
6. **Every adapter returns `[N]` on a `DayBatch`** (last cell, with `CLEAN_DATA = True`: 28 DJI stocks, `PRG.US` discarded). Together with the checks above, this meets the Step 2 criterion *all six adapters pass the equivalence check*. FactorVAE's run is the slowest (about 5 minutes), because of FinBench's Qlib sampler on the reference side.